In [1]:
import pandas as pd
import json

In [2]:
brand_list = pd.read_csv('../EDA/brand_levels.csv')
brand_list.head()

,brand,num_brand,brand_level
0,CoverGirl,1558,Major
1,Nyx,1403,Major
2,ColourPop,1270,Major
3,Tarte,1119,Major
4,Sephora,1095,Major


In [3]:
# I started with this, so I have to keep it unless I want to have to undo a bunch of things
single_brand = ['Nyx', 'bareMinerals', 'Bareminerals', 'Antonym', 'Axiology', 'BECCA', 'Zao', 'Sheglam', 'Aziza', 'e.l.f.', 'about-face', 'Revolution', 'Revlon', 'Tarte', 'Sisley', 'Profusion', 'Pixi', 'Pacifica', 'Milani', 'Nudestix', 'Maybelline', 'Lique', 'Kindred', 'Kaja', 'ILIA', 'Flower', 'Glossier', 'Essence', 'E.l.f.', 'Covergirl', 'CoverGirl', 'Doucce', 'Colourpop', 'Clinique', 'Cheekbone', 'Catrice', 'Caliray', 'COVERGIRL', 'Benefit', 'withSimplicity', 'Almay', 'Smashbox', 'Pur', 'PUR', 'Omiana', 'No7', 'Neutrogena', 'NYX', 'NARS', 'Morphe', 'MadHippie', 'M.A.C', 'Lumin', 'Kosas', 'Kokie', 'Joah', 'Givenchy', 'Gabriel', 'Erborian', 'ColourPop', 'Beautycounter', 'Rimmel', 'tarte', 'Ultabeauty', 'Ardell', 'Baeblu', 'Thread', 'Sorme', 'SHEGLAM', 'Rubies', 'Palladio', 'Olay', 'Mcobeauty.', 'M.a.c', 'Uoma', 'Sassy+Chic', 'No7,']

# Importing the brands fix file
brands = pd.read_csv('../EDA/brand_fixing.csv')
brands.drop(columns=['count'], inplace=True)
brand_dict = dict(zip(brands['bad'], brands['good']))
brands.head()

,bad,good
0,18. 21,18.21
1,18.21 Age,18.21
2,18.21 Man,18.21
3,18.21 Octane,18.21
4,18.21Man Made,18.21


In [4]:
def get_brand(name):
    temp = name.split(' ')
    if temp[0] in single_brand:
        return temp[0]
    try:
        temp2 = temp[0] + ' ' + temp[1]
    except:
        return temp[0]
    return temp2

def fix_brand(brand):
    if brand in brand_dict.keys():
        return brand_dict[brand]
    return brand

def get_ingredients_num(ingredients):
    try:
        return ingredients.count(',') + 1
    except:
        return -1

def full_run(input_file_names, input_file_locations, file_group):
    output_dfs = []
    for index, file_name in enumerate(input_file_names):
        try:
            final_file_name = input_file_locations[index] + file_name + '_products.json'
            with open(final_file_name, 'r') as file:
                json_data = json.load(file)
                
            data = [{'name': name, 'url': info['url'], 'ingredients': info['ingredients']} for name, info in json_data.items()]

            df = pd.DataFrame(data)
            df['Brand 2'] = df['name'].apply(get_brand)
            df['brand'] = df['Brand 2'].apply(fix_brand)
            df.drop(['Brand 2'], axis=1, inplace=True)
            df = df.merge(brand_list, on='brand', how='left')
            df['group'] = file_name
            df['super_group'] = file_group
            
            df['ingred_num'] = df['ingredients'].apply(get_ingredients_num)
            
            
            # Add it to the big df before 
            output_dfs.append(df)
            
            output_file_name = './UPDATED_' + file_group + '/UPDATED_' + file_name + '_products.json'
            df = df.set_index('name').to_dict(orient='index')
            with open(output_file_name, 'w') as json_file:
                json.dump(df, json_file, indent=4)

            print(f"Data has been written to {output_file_name}")
            
            #print(file_name, df.shape)
        except Exception as e:
            print(f'Problem with {file_name}')
            print(e)
    final_df = pd.concat(output_dfs, ignore_index = True)
    #print(final_df.columns)
    final_df = final_df.drop_duplicates(subset='name')
    final_df2 = final_df.set_index('name').to_dict(orient='index')
    #print(final_df.columns)
    output_file_name = './UPDATED_' + file_group + '/' + file_group + '_FULL_GROUP.json'
    with open(output_file_name, 'w') as json_file:
        json.dump(final_df2, json_file, indent=4)

    print(f"Data has been written to {output_file_name}")
    return final_df
            
    

In [5]:
group1 = 'OTHER'
group1_input_file_names = ['Glitter']
group1_input_file_locations = ['./Makeup/']
group_other = full_run(group1_input_file_names, group1_input_file_locations, group1)

Data has been written to ./UPDATED_OTHER/UPDATED_Glitter_products.json
Data has been written to ./UPDATED_OTHER/OTHER_FULL_GROUP.json


In [6]:
group2 = 'LIPS'
group2_input_file_names = ['Lip_balm', 'Lip_gloss', 'Lip_liner', 'Lip_plumper', 'lip+balm+with+SPF', 'Lipstick']
group2_input_file_locations = ['./Makeup/', './Makeup/', './Makeup/', './Makeup/', './Makeup/', './Makeup/']
group_lips = full_run(group2_input_file_names, group2_input_file_locations, group2)

Data has been written to ./UPDATED_LIPS/UPDATED_Lip_balm_products.json
Data has been written to ./UPDATED_LIPS/UPDATED_Lip_gloss_products.json
Data has been written to ./UPDATED_LIPS/UPDATED_Lip_liner_products.json
Data has been written to ./UPDATED_LIPS/UPDATED_Lip_plumper_products.json
Data has been written to ./UPDATED_LIPS/UPDATED_lip+balm+with+SPF_products.json
Data has been written to ./UPDATED_LIPS/UPDATED_Lipstick_products.json
Data has been written to ./UPDATED_LIPS/LIPS_FULL_GROUP.json


In [7]:
group3 = 'EYES'
group3_input_file_names = ['Eye_liner', 'Eye_makeup_remover', 'Eye_shadow', 'Eyelash_glue', 'Mascara', 'Other_eye_makeup', 'Around-eye_cream']
group3_input_file_locations = ['./Makeup/','./Makeup/','./Makeup/','./Makeup/','./Makeup/','./Makeup/','./Face_And_Body/']
group_eyes = full_run(group3_input_file_names, group3_input_file_locations, group3)

Data has been written to ./UPDATED_EYES/UPDATED_Eye_liner_products.json
Data has been written to ./UPDATED_EYES/UPDATED_Eye_makeup_remover_products.json
Data has been written to ./UPDATED_EYES/UPDATED_Eye_shadow_products.json
Data has been written to ./UPDATED_EYES/UPDATED_Eyelash_glue_products.json
Data has been written to ./UPDATED_EYES/UPDATED_Mascara_products.json
Data has been written to ./UPDATED_EYES/UPDATED_Other_eye_makeup_products.json
Data has been written to ./UPDATED_EYES/UPDATED_Around-eye_cream_products.json
Data has been written to ./UPDATED_EYES/EYES_FULL_GROUP.json


In [8]:
group4 = 'FACE'
group4_input_file_names = ['BB_Cream', 'CC_Cream', 'Facial_cleanser', 'Facial_moisturizer__treatment', 'Mask', 'Oil_controller', 'Pore_strips', 'Serums_&_Essences', 'Skin_fading__lightener', 'Toners__astringents', 'Blush', 'Bronzer_Highlighter', 'Brow_liner', 'Concealer', 'Facial_powder', 'Foundation', 'Makeup_primer', 'Makeup_remover']
group4_input_file_locations = ['./Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Makeup/', './Makeup/', './Makeup/', './Makeup/', './Makeup/', './Makeup/', './Makeup/', './Makeup/']
group_face = full_run(group4_input_file_names, group4_input_file_locations, group4)

Data has been written to ./UPDATED_FACE/UPDATED_BB_Cream_products.json
Data has been written to ./UPDATED_FACE/UPDATED_CC_Cream_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Facial_cleanser_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Facial_moisturizer__treatment_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Mask_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Oil_controller_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Pore_strips_products.json
Problem with Serums_&_Essences
'name'
Data has been written to ./UPDATED_FACE/UPDATED_Skin_fading__lightener_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Toners__astringents_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Blush_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Bronzer_Highlighter_products.json
Data has been written to ./UPDATED_FACE/UPDATED_Brow_liner_products.json
Data has been written to ./UPDATED_FACE/UP

In [9]:
group5 = 'BODY'
group5_input_file_names = ['After_sun_product', 'Antiperspirant__deodorant', 'Bar_soap', 'Bath_oil__salts__soak', 'Body_firming_lotion', 'Body_oil', 'Body_wash__cleanser', 'Bubble_bath', 'Exfoliant__scrub', 'Foot_cleansing', 'Foot_moisturizer', 'Foot_odor_control', 'Hand_cream', 'Hand_sanitizer', 'Liquid_hand_soap', 'Moisturizer', 'Muscle__joint_soreness', 'Body_art']
group5_input_file_locations = ['./Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Face_And_Body/', './Makeup/']
group_body = full_run(group5_input_file_names, group5_input_file_locations, group5)

Data has been written to ./UPDATED_BODY/UPDATED_After_sun_product_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Antiperspirant__deodorant_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Bar_soap_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Bath_oil__salts__soak_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Body_firming_lotion_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Body_oil_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Body_wash__cleanser_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Bubble_bath_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Exfoliant__scrub_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Foot_cleansing_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Foot_moisturizer_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Foot_odor_control_products.json
Data has been written to ./UPDATED_BODY/UPDATED_Hand_

In [10]:
combined_df = pd.concat([group_other, group_lips, group_body, group_eyes, group_face])
#combined_df.to_csv('./EVERYTHING.csv', index=False)
combined_df.head()

,name,url,ingredients,brand,num_brand,brand_level,group,super_group,ingred_num
0,"Milk Makeup Glitter Stick, Techno (2020 formul...",https://www.ewg.org/skindeep/products/930057-M...,": Ricinus communis (castor) seed oil, polyethy...",Milk Makeup,250.0,Medium,Glitter,OTHER,18
1,Profusion Cosmetics Glow Up! Multi Dimensional...,https://www.ewg.org/skindeep/products/922596-P...,"LIQUID GLITTER INGREDIENTS: AQUA, MICA, TRIDEC...",Profusion,217.0,Medium,Glitter,OTHER,18
2,"JD Glow Cosmetics Glitter Tube, Goddess (2020 ...",https://www.ewg.org/skindeep/products/943914-J...,": Polyethylene Terephalate, Acrylates, Denatur...",JD Glow,142.0,Medium,Glitter,OTHER,8
3,"L.a. Colors Glitter Palette, Cgp694 Delightful...",https://www.ewg.org/skindeep/products/884475-L...,"MINERAL OIL (PARAFFINUM LIQUIDUM), POLYISOBUTE...",L.A. Colors,606.0,Major,Glitter,OTHER,22
4,"Tood Beauty bioglitter, jasper",https://www.ewg.org/skindeep/products/1000467-...,"Isododecane, Rayon, Hydrogenated Styrene/isopr...",Tood Beauty,12.0,Minor,Glitter,OTHER,21


0        Milk Makeup Glitter Stick, Techno (2020 formul...
1        Profusion Cosmetics Glow Up! Multi Dimensional...
2        JD Glow Cosmetics Glitter Tube, Goddess (2020 ...
3        L.a. Colors Glitter Palette, Cgp694 Delightful...
4                           Tood Beauty bioglitter, jasper
                               ...                        
33600    Revolution Gentle Eye Makeup Remover, Peach Ke...
33601    Red Earth Green Rush Bubble Cleanser  (2018 fo...
33602                    Erborian Cleansing Micellar Water
33603    Studio Selection Makeup Removing Facial Towele...
33604       artNaturals Makeup Remover  (2020 formulation)
Name: name, Length: 83245, dtype: object

In [25]:
def remove_things(name):
    problems = [',', '-', '(', ')', ',']
    for prob in problems:
        try:
            name = name.replace(prob, '')
        except:
            pass
    return name

In [ ]:
names_df = combined_df[['name']]
names_df['name'] = names_df['name'].apply(remove_things)
names_df['words'] = names_df['name'].str.lower().str.split()



names_df

/var/folders/cv/ywgm4kr10_70hyq9c73zhw400000gn/T/ipykernel_59059/330874564.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  names_df['words'] = names_df['name'].str.lower().str.split()
/var/folders/cv/ywgm4kr10_70hyq9c73zhw400000gn/T/ipykernel_59059/330874564.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  names_df['name'] = names_df['name'].apply(remove_things)


,name,words
0,Milk Makeup Glitter Stick Techno 2020 formulation,"[milk, makeup, glitter, stick,, techno, (2020,..."
1,Profusion Cosmetics Glow Up! Multi Dimensional...,"[profusion, cosmetics, glow, up!, multi, dimen..."
2,JD Glow Cosmetics Glitter Tube Goddess 2020 fo...,"[jd, glow, cosmetics, glitter, tube,, goddess,..."
3,L.a. Colors Glitter Palette Cgp694 Delightful ...,"[l.a., colors, glitter, palette,, cgp694, deli..."
4,Tood Beauty bioglitter jasper,"[tood, beauty, bioglitter,, jasper]"
...,...,...
33600,Revolution Gentle Eye Makeup Remover Peach Ker...,"[revolution, gentle, eye, makeup, remover,, pe..."
33601,Red Earth Green Rush Bubble Cleanser 2018 for...,"[red, earth, green, rush, bubble, cleanser, (2..."
33602,Erborian Cleansing Micellar Water,"[erborian, cleansing, micellar, water]"
33603,Studio Selection Makeup Removing Facial Towele...,"[studio, selection, makeup, removing, facial, ..."


In [27]:
word_count_dict = {}

for _, row in names_df.iterrows():
    for word in set(row['words']):
        if word not in word_count_dict:
            word_count_dict[word] = 0
        word_count_dict[word] += 1
        
word_counts = pd.DataFrame(list(word_count_dict.items()), columns=['word', 'count'])

word_counts.sort_values(by='count', ascending=False)

,word,count
2,formulation),28392
19,(2019,12242
1,(2020,11073
600,lip,9084
75,&,8874
...,...,...
16252,lavant,1
16255,kikkerland,1
16256,"meadows,",1
16257,"pod,",1


In [28]:
word_counts.to_csv('./words_present.csv', index=False)